In [1]:
!pip install gymnasium

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)

   ------------- -------------------------- 1/3 [cloudpickle]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ---

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import matplotlib.animation as animation
from IPython.display import HTML

# Define the Maze Grid (0: Empty, 1: Wall)
# 12x12 grid
maze = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1],
    [1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1],
    [1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1],
    [1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    [1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1],
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1],
    [1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
])

# Victims positions (y, x) and names
victims = {
    'V1': (1, 10),
    'V2': (5, 9),
    'V3': (9, 3),
    'V4': (10, 6)
}

# Initial positions of the squad agents
agents = {
    'Bot_Alpha': (1, 1),
    'Bot_Beta': (10, 10)
}

def bfs_shortest_path(maze, start, goal):
    """Finds the shortest path from start to goal in the maze using BFS."""
    if start == goal:
        return [start]

    queue = deque([[start]])
    visited = {start}
    height, width = maze.shape

    while queue:
        path = queue.popleft()
        curr = path[-1]

        y, x = curr
        # Up, Down, Left, Right
        neighbors = [(y-1, x), (y+1, x), (y, x-1), (y, x+1)]

        for ny, nx in neighbors:
            if 0 <= ny < height and 0 <= nx < width:
                if maze[ny, nx] == 0 and (ny, nx) not in visited:
                    new_path = list(path) + [(ny, nx)]
                    if (ny, nx) == goal:
                        return new_path
                    visited.add((ny, nx))
                    queue.append(new_path)
    return [] # No path found


In [3]:
def assign_targets(agents_pos, remaining_victims, maze):
    """Assigns each agent to the closest unassigned victim based on path length."""
    assignments = {}
    assigned_victims = set()

    for agent_name, agent_pos in agents_pos.items():
        best_victim = None
        shortest_dist = float('inf')
        best_path = []

        for vic_name, vic_pos in remaining_victims.items():
            if vic_name in assigned_victims:
                continue
            path = bfs_shortest_path(maze, agent_pos, vic_pos)
            if path and len(path) < shortest_dist:
                shortest_dist = len(path)
                best_victim = vic_name
                best_path = path

        if best_victim:
            assignments[agent_name] = {
                'victim': best_victim,
                'pos': remaining_victims[best_victim],
                'path': best_path
            }
            assigned_victims.add(best_victim)

    return assignments


In [4]:
# Initialize simulation states
current_agents = dict(agents)
print(current_agents)
remaining_victims = dict(victims)
print(remaining_victims)

# History tracking for visualization
history_agents = {'Bot_Alpha': [current_agents['Bot_Alpha']], 'Bot_Beta': [current_agents['Bot_Beta']]}
history_victims = [dict(remaining_victims)]

# Run simulation loop until all victims are rescued
step = 0
max_steps = 100

active_assignments = assign_targets(current_agents, remaining_victims, maze)

while remaining_victims and step < max_steps:
    print(f"--- Step {step} ---")
    for agent_name, pos in current_agents.items():
        assignment_info = active_assignments.get(agent_name)
        target_info = f"Targeting {assignment_info['victim']}" if assignment_info else "Idle"
        print(f"  {agent_name} is at {pos} -> {target_info}")

    # Move each agent one step forward along its path
    for agent_name in list(current_agents.keys()):
        if agent_name in active_assignments:
            path = active_assignments[agent_name]['path']
            if len(path) > 1:
                # Update position to next step on path
                current_agents[agent_name] = path[1]
                active_assignments[agent_name]['path'] = path[1:]
            else:
                # Arrived at victim's destination
                rescued_vic = active_assignments[agent_name]['victim']
                if rescued_vic in remaining_victims:
                    print(f"  >> SUCCESS: {agent_name} rescued {rescued_vic} at {current_agents[agent_name]}!")
                    del remaining_victims[rescued_vic]
        else:
            # Idle if no assignment
            pass

    # Record history
    for agent_name in current_agents.keys():
        history_agents[agent_name].append(current_agents[agent_name])
    history_victims.append(dict(remaining_victims))

    # Recalculate assignments for next cycle
    active_assignments = assign_targets(current_agents, remaining_victims, maze)
    step += 1

print(f"\nSimulation finished in {step} steps.\n")

# Visualization Setup
fig, ax = plt.subplots(figsize=(8, 8))

def update(frame):
    ax.clear()
    # Draw Maze
    ax.imshow(maze, cmap='binary_r')

    # Draw Grid boundaries
    ax.set_xticks(np.arange(-0.5, 12, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 12, 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)

    # Draw Remaining Victims
    frame_victims = history_victims[min(frame, len(history_victims)-1)]
    for name, pos in frame_victims.items():
        ax.plot(pos[1], pos[0], 'ro', markersize=12, label='Victim' if name == list(frame_victims.keys())[0] else "")
        ax.text(pos[1], pos[0] - 0.4, name, color='red', weight='bold', ha='center')

    # Draw Agents
    for idx, (agent_name, color) in enumerate([('Bot_Alpha', 'blue'), ('Bot_Beta', 'orange')]):
        pos = history_agents[agent_name][min(frame, len(history_agents[agent_name])-1)]
        ax.plot(pos[1], pos[0], marker='s', color=color, markersize=14, label=agent_name)
        ax.text(pos[1], pos[0] + 0.5, agent_name, color=color, weight='bold', ha='center')

    ax.set_title(f"Rescue Bot Squad Simulation - Step {frame}")
    ax.legend(loc='upper right')
    ax.axis('off')

total_frames = max(len(history_agents['Bot_Alpha']), len(history_agents['Bot_Beta']))
anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=600, repeat=False)
plt.close()

# Render animation in Colab
HTML(anim.to_jshtml())


{'Bot_Alpha': (1, 1), 'Bot_Beta': (10, 10)}
{'V1': (1, 10), 'V2': (5, 9), 'V3': (9, 3), 'V4': (10, 6)}
--- Step 0 ---
  Bot_Alpha is at (1, 1) -> Targeting V1
  Bot_Beta is at (10, 10) -> Targeting V2
--- Step 1 ---
  Bot_Alpha is at (1, 2) -> Targeting V1
  Bot_Beta is at (9, 10) -> Targeting V2
--- Step 2 ---
  Bot_Alpha is at (1, 3) -> Targeting V1
  Bot_Beta is at (8, 10) -> Targeting V2
--- Step 3 ---
  Bot_Alpha is at (2, 3) -> Targeting V1
  Bot_Beta is at (7, 10) -> Targeting V2
--- Step 4 ---
  Bot_Alpha is at (3, 3) -> Targeting V1
  Bot_Beta is at (7, 9) -> Targeting V2
--- Step 5 ---
  Bot_Alpha is at (3, 4) -> Targeting V1
  Bot_Beta is at (6, 9) -> Targeting V2
--- Step 6 ---
  Bot_Alpha is at (3, 5) -> Targeting V1
  Bot_Beta is at (5, 9) -> Targeting V2
  >> SUCCESS: Bot_Beta rescued V2 at (5, 9)!
--- Step 7 ---
  Bot_Alpha is at (2, 5) -> Targeting V1
  Bot_Beta is at (5, 9) -> Targeting V4
--- Step 8 ---
  Bot_Alpha is at (1, 5) -> Targeting V1
  Bot_Beta is at (6, 9)